# Alternative splicing from RNA-seq data

This mini-protocol produces normalized, gene-annotated splicing phenotypes for splicing-QTL analysis using either LeafCutter or psichomics.

#### Miniprotocol Timing

Timing: TBD

## Overview

The [splicing-calling module](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/calling/splicing_calling.html) provides two alternative quantification routes. LeafCutter derives intron excision ratios from STAR splice-junction files without relying on a transcript annotation. psichomics calculates percent-spliced-in (PSI) values for annotated alternative-splicing events and therefore also requires the SUPPA annotation object.

The [normalization module](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/QC/splicing_normalization.html) performs missingness and variability filtering followed by quantile normalization. The [gene-annotation module](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html) then assigns coordinates and phenotype groups required by TensorQTL. LeafCutter and psichomics are alternative routes, not one combined chain.

## Steps

| **Analysis goal** | **Commands to run, in order** | **Inputs** |
| --- | --- | --- |
| LeafCutter from aligned RNA-seq data | 1 → 2 → 3 | `output/rnaseq/protocol_example.rnaseq.bam.list.txt`; STAR/WASP alignment directories |
| LeafCutter from a precomputed ratio matrix | 2 → 3 | `input/rnaseq/protocol_example.leafcutter.intron_usage_perind.counts.gz`; `input/rnaseq/protocol_example.leafcutter.intron_count.tsv` |
| psichomics from aligned RNA-seq data | 4 → 5 → 6 | BAM list and alignments above; `input/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.ERCC.SUPPA_annotation.rds` |
| psichomics from a precomputed PSI matrix | 5 → 6 | `input/rnaseq/protocol_example.psichomics.psi_raw_data.tsv` |

Within the selected route, run the commands in numerical order. The bundled example data support the two precomputed-matrix routes; the calling routes require aligned BAM files.

### [1. Quantify intron usage with LeafCutter](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/calling/splicing_calling.html)

**What it does:** `leafcutter` extracts splice junctions and clusters introns to calculate per-sample intron excision ratios.

In [ ]:
sos run pipeline/splicing_calling.ipynb leafcutter   --cwd output/splicing/leafcutter   --samples output/rnaseq/protocol_example.rnaseq.bam.list.txt   --data-dir output/rnaseq/star_output_wasp

### [2. Normalize LeafCutter ratios](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/QC/splicing_normalization.html)

**What it does:** `leafcutter_norm` filters introns and clusters, mean-imputes retained missing values, and quantile-normalizes the ratio matrix.

In [ ]:
sos run pipeline/splicing_normalization.ipynb leafcutter_norm   --cwd output/splicing/leafcutter   --ratios <path/to/protocol_example.leafcutter.intron_usage_perind.counts.gz>   --mean-impute

### [3. Annotate LeafCutter phenotypes](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html)

**What it does:** `annotate_leafcutter_isoforms` maps introns to genes and writes the coordinate-sorted phenotype matrix and phenotype-group file used by TensorQTL.

In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_leafcutter_isoforms   --cwd output/splicing/leafcutter   --phenoFile <path/to/protocol_example.leafcutter.intron_usage_perind.counts.gz_raw_data.qqnorm.txt>   --intron-count tests/fixtures/gene_annotation/protocol_example.leafcutter.intron_count.tsv   --coordinate-annotation tests/fixtures/gene_annotation/Homo_sapiens.GRCh38.103.collapse_only.gene.chr22.gtf.gz   --map-stra site

### [4. Quantify annotated events with psichomics](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/calling/splicing_calling.html)

**What it does:** `psichomics` uses aligned reads and a SUPPA event annotation to calculate PSI values for alternative-splicing events.

In [ ]:
sos run pipeline/splicing_calling.ipynb psichomics   --cwd output/splicing/psichomics   --samples output/rnaseq/protocol_example.rnaseq.bam.list.txt   --data-dir output/rnaseq/star_output_wasp   --splicing-annotation <path/to/Homo_sapiens.GRCh38.103.chr.reformatted.ERCC.SUPPA_annotation.rds>

### [5. Normalize psichomics PSI values](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/QC/splicing_normalization.html)

**What it does:** `psichomics_norm` filters PSI events for missingness and variability and produces a quantile-normalized event matrix.

In [ ]:
sos run pipeline/splicing_normalization.ipynb psichomics_norm   --cwd output/splicing/psichomics   --ratios <path/to/protocol_example.psichomics.psi_raw_data.tsv>

### [6. Annotate psichomics phenotypes](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html)

**What it does:** `annotate_psichomics_isoforms` assigns genomic coordinates and phenotype groups to normalized psichomics events for TensorQTL.

In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_psichomics_isoforms   --cwd output/splicing/psichomics   --phenoFile output/splicing/psichomics/psichomics_raw_data_bedded.qqnorm.txt   --coordinate-annotation tests/fixtures/gene_annotation/Homo_sapiens.GRCh38.103.collapse_only.gene.chr22.gtf.gz

## Output Files

| Route/step | Output filename and relative path | Description |
| --- | --- | --- |
| LeafCutter calling | `output/splicing/leafcutter/*_intron_usage_perind.counts.gz`; `*_intron_usage_perind_numers.counts.gz` | Intron excision ratios and supporting intron counts |
| LeafCutter normalization | `input/rnaseq/protocol_example.leafcutter.intron_usage_perind.counts.gz_raw_data.txt`; `input/rnaseq/protocol_example.leafcutter.intron_usage_perind.counts.gz_raw_data.qqnorm.txt` | QC-filtered and quantile-normalized ratios |
| LeafCutter annotation | `output/splicing/leafcutter/protocol_example.leafcutter.intron_usage_perind.counts.gz_raw_data.qqnorm.formated.bed.gz`; `output/splicing/leafcutter/protocol_example.leafcutter.intron_usage_perind.counts.gz_raw_data.qqnorm.phenotype_group.txt` | TensorQTL phenotype and grouping files |
| psichomics calling | `output/splicing/psichomics/psi_raw_data.tsv` | Raw event-level PSI matrix |
| psichomics normalization | `output/splicing/psichomics/psichomics_raw_data_bedded.qqnorm.txt` | QC-filtered and normalized PSI matrix |
| psichomics annotation | `output/splicing/psichomics/psichomics_raw_data_bedded.qqnorm.formated.bed.gz`; `output/splicing/psichomics/psichomics_raw_data_bedded.qqnorm.phenotype_group.txt` | TensorQTL phenotype and grouping files |

## Anticipated Results

Each selected route ends with a coordinate-sorted, bgzip-compressed phenotype matrix and a matching phenotype-group file. Rows represent introns or annotated splicing events, columns represent samples, and the normalized values can be supplied directly to splicing-QTL association testing.

## Command interface

In [ ]:
sos run pipeline/splicing_calling.ipynb -h

In [ ]:
sos run pipeline/splicing_normalization.ipynb -h

In [ ]:
sos run pipeline/gene_annotation.ipynb -h